In [1]:
import os

# Adjust path based on what `ls /usr/lib/jvm/` actually shows
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-amazon-corretto.x86_64"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DAT204M-EDA-Sagemaker")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.InstanceProfileCredentialsProvider")
    .config("spark.driver.memory", "10g")
    .getOrCreate()
)

print("Spark ready:", spark.version)

:: loading settings :: url = jar:file:/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b4d50efc-a974-4919-9a90-e920f717b920;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 258ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded

26/07/05 16:02:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark ready: 3.3.0


In [2]:
S3_PATH = "s3a://dat204m-project-g3/cleaned_data_final/"

# Just read the schema/plan first — this is lazy, no actual data pulled yet
df = spark.read.parquet(S3_PATH)

print(f"Column count: {len(df.columns)}")
print(f"Columns: {df.columns}\n")

df.printSchema()

26/07/05 16:03:48 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Column count: 11
Columns: ['user_id', 'stime', 'session_id', 'item_id', 'event_id', 'item_condition_name', 'product_id', 'brand_name', 'size_name', 'color', 'shipper_name']

root
 |-- user_id: long (nullable = true)
 |-- stime: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- item_id: long (nullable = true)
 |-- event_id: string (nullable = true)
 |-- item_condition_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- brand_name: string (nullable = true)
 |-- size_name: string (nullable = true)
 |-- color: string (nullable = true)
 |-- shipper_name: string (nullable = true)



In [3]:
# Pull a small number of rows to confirm actual data read works (not just metadata)
df.show(5, truncate=False)

[Stage 1:>                                                          (0 + 1) / 1]

+-------+-------------------+----------------------------------------------------------------+---------+---------+-------------------+----------+----------------------+-----------+-----+------------+
|user_id|stime              |session_id                                                      |item_id  |event_id |item_condition_name|product_id|brand_name            |size_name  |color|shipper_name|
+-------+-------------------+----------------------------------------------------------------+---------+---------+-------------------+----------+----------------------+-----------+-----+------------+
|6312169|2023-05-25 03:05:29|0e12ab0fc425f337d2ffb6df45c1fdfe99eef4eb0893138e8d9785bfd8ada5b1|231439122|item_view|New                |436_1224  |Air Jordan            |10.5 (43.5)|null |Buyer       |
|6312522|2023-05-08 02:10:54|4fea2b3136031b1a2a465a361022e3df2396ca541de0855b89546234c1dd124e|216873628|item_view|New                |0_561     |null                  |null       |null |Buyer       |
